In [ ]:
# Importing Libraries

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,CSVLogger
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Loading Dataset

train_path = "/kaggle/input/datasets/afrojkhan0220/leaf-nonleaf-image/Leaf-Nonleaf_Image/train"  
valid_path= "/kaggle/input/datasets/afrojkhan0220/leaf-nonleaf-image/Leaf-Nonleaf_Image/valid"  
test_path = "/kaggle/input/datasets/afrojkhan0220/leaf-nonleaf-image/Leaf-Nonleaf_Image/test" 

In [ ]:
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [ ]:
IMAGE_SIZE = (380,380)
BATCH_SIZE = 32

In [ ]:
# Image Data Generator

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range      = 40,
    zoom_range          = 0.5,
    width_shift_range   = 0.3,
    height_shift_range  = 0.3,
    brightness_range    = [0.7, 1.4],
    horizontal_flip     = True,
    vertical_flip       = True,
    shear_range         = 0.3,
    channel_shift_range = 10.0,
    fill_mode           = "nearest"
)

In [ ]:
val_test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
# Training Image Preprocessing

train = train_gen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42

)

In [ ]:
# Validation Image Preprocessing

val   = val_test_gen.flow_from_directory(
    valid_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False)

In [ ]:
# Test Image Preprocessing

test  = val_test_gen.flow_from_directory(
    test_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False)

In [ ]:
# Fix non__leaf → Non__Leaf in class indices
class_names = list(train.class_indices.keys())
NUM_CLASSES = len(class_names)

if 'non__leaf' in train.class_indices:
    idx = train.class_indices['non__leaf']
    train.class_indices['Non__Leaf'] = idx
    del train.class_indices['non__leaf']
    
    # Fix class_names list
    class_names = [name.replace('non__leaf', 'Non__Leaf') 
                   for name in class_names]
    
    print("Fixed: non__leaf → Non__Leaf")
    print(f"Non__Leaf index: {train.class_indices['Non__Leaf']}")



with open("class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

In [ ]:
train_labels = train.classes 
unique_labels = np.unique(train_labels)


class_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_labels,
    y=train_labels
)


class_weight_dict = dict(zip(unique_labels, class_weights))


print(f"Total classes found: {len(unique_labels)}")
print(f"Weight for Class 0 (Apple Scab): {class_weight_dict[0]:.4f}")


if 60 in class_weight_dict:
    print(f"Weight for Class 60 (NON-LEAF): {class_weight_dict[60]:.4f}")

In [ ]:
# Data Agumentation

images, labels = next(train)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for i, ax in enumerate(axes.flat):
    # --- THE FIX ---
    # Rescale the preprocessed image back to 0-1 just for the eyes
    img_to_show = (images[i] - images[i].min()) / (images[i].max() - images[i].min())
    
    ax.imshow(img_to_show)
    ax.set_title(class_names[np.argmax(labels[i])], fontsize=7)
    ax.axis("off")

plt.suptitle("Augmented Training Samples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Building pretrained model

base_model = EfficientNetB4(weights='imagenet',
                            include_top=False,
                            input_shape=(380,380, 3))
base_model.trainable = False 

In [ ]:
# Building Dense Layer

x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(train.num_classes, activation='softmax', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=outputs)

In [ ]:
# Optimizing Phase 1

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    ModelCheckpoint("best_model.keras",
                    monitor="val_accuracy",
                    save_best_only=True,
                    mode="max",
                    verbose=1),
    EarlyStopping(monitor="val_accuracy",
                  patience=5,
                  restore_best_weights=True,
                  verbose=1),
    ReduceLROnPlateau(monitor="val_loss",
                      factor=0.2,
                      patience=2,
                      min_lr=1e-7,
                      verbose=1),
    CSVLogger("training_log.csv", append=True)
]

In [ ]:
# Training Phase 1

history1 = model.fit(
    train,
    steps_per_epoch=400, 
    validation_data=val,
    validation_steps=100,
    epochs=10,
    callbacks=callbacks,
    class_weight = class_weight_dict
)

In [ ]:
# Visualization of phase 1 Accuracy


epochs = range(1, len(history1.history['accuracy']) + 1)
plt.plot(epochs,history1.history['accuracy'],color='red',label='Training Accuracy')
plt.plot(epochs,history1.history['val_accuracy'],color='blue',label='Training Validation Accuracy')
plt.xlabel('No. Of Epochs')
plt.ylabel('Accuracy result')
plt.title('Visualizaion of Accuracy Result')
plt.legend()
plt.show()

In [ ]:
# Visualization of phase 1 Loss

epochs = range(1, len(history1.history['loss']) + 1)
plt.plot(epochs,history1.history['loss'],color='red',label='Training Loss')
plt.plot(epochs,history1.history['val_loss'],color='blue',label='Training Validation Loss')
plt.xlabel('No. Of Epochs')
plt.ylabel('Loss result')
plt.title('Visualizaion of Loss Result')
plt.legend()
plt.show()

In [ ]:
# train last 100 layers

base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

In [ ]:
# Optimizing Phase 2

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    ModelCheckpoint("best_model.keras",
                    monitor="val_accuracy",
                    save_best_only=True,
                    mode="max",
                    verbose=1),
    EarlyStopping(monitor="val_accuracy",
                  patience=3,
                  restore_best_weights=True,
                  verbose=1),
    ReduceLROnPlateau(monitor="val_loss",
                      factor=0.3,
                      patience=5,
                      min_lr=1e-7,
                      verbose=1),
    CSVLogger("training_log.csv", append=True)
]

In [ ]:
# Training Phase 2 model

history2 = model.fit(
    train,
    steps_per_epoch=400, 
    validation_data=val,
    validation_steps = 100,
    epochs=25,
    callbacks=callbacks,
    class_weight = class_weight_dict
)
model_best = max(history2.history['val_accuracy'])*100

In [ ]:
# Saving Model

model.save("model.keras")
model.save("model.h5")

In [ ]:
# model evaluation on training set

train_loss,train_acc=model.evaluate(train)
print(train_loss,train_acc)

In [ ]:
# Assuming your generator is named 'train_generator'
indices = train.class_indices
labels = list(indices.keys())

print("--- MODEL CLASS MAPPING ---")
for name, index in indices.items():
    # Apply our 'Pretty Name' logic
    if "non__leaf" in name.lower():
        display_name = "NOT A LEAF"
    else:
        display_name = name.replace("___", " - ").replace("_", " ")
    
    # Print in UPPERCASE
    print(f"Index {index}: {display_name.upper()}")

In [ ]:
# model evaluation on validation set

val_loss,val_acc = model.evaluate(val)
print(val_loss,val_acc)

In [ ]:
# Visualization of phase 2 Accuracy

epochs = range(1, len(history2.history['accuracy']) + 1)
plt.plot(epochs,history2.history['accuracy'],color='red',label='Training Accuracy')
plt.plot(epochs,history2.history['val_accuracy'],color='blue',label='Training Validation Accuracy')
plt.xlabel('No. Of Epochs')
plt.ylabel('Accuracy result')
plt.title('Visualizaion of Accuracy Result')
plt.legend()
plt.show()

In [ ]:
# Visualization of phase2 2 Loss

epochs = range(1, len(history2.history['loss']) + 1)
plt.plot(epochs,history2.history['loss'],color='red',label='Training Loss')
plt.plot(epochs,history2.history['val_loss'],color='blue',label='Training Validation Loss')
plt.xlabel('No. Of Epochs')
plt.ylabel('Loss result')
plt.title('Visualizaion of Loss Result')
plt.legend()
plt.show()

In [ ]:
y_pred = model.predict(test)
print(y_pred,y_pred.shape)

In [ ]:
predict_categories = tf.argmax(y_pred,axis=1)
print(predict_categories)

In [ ]:
y_true = test.classes
print(y_true)

In [ ]:
class_name = list(test.class_indices.keys())

In [ ]:

print(classification_report(y_true,predict_categories,target_names=class_name))


In [ ]:
cm = confusion_matrix(y_true,predict_categories)
print(cm)

In [ ]:
plt.figure(figsize=(80,80))
sns.heatmap(cm,annot=True,annot_kws={'size':25})
plt.xlabel('Predicated Class',fontsize=20)
plt.ylabel('Actual_class',fontsize=20)
plt.title('Plant Disease Predication Confusion Matrix',fontsize=30)
plt.show()
